# Naive model

Main idea: prediction on a particular day = average of all readings on that day before. 

If there are no readings on the same day before, do forward fill.


In [ ]:
#import everything!
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, datetime, timedelta
from itertools import product
from copy import deepcopy
from PreRun import PreRun, PostRun
from sklearn.metrics import mean_absolute_error, mean_squared_error

#path to data
read_path = "../../../../data_ds_project/parquet_cleaned_energy"
#systems
good_systems_list = [4, 10, 33, 36, 50, 51, 1199, 1204, 1283, 1284, 1289, 1332, 4902, 4903]
reader_types = ["meter", "inverter", None]
#systems_cleaned
systems_cleaned = pd.read_csv("../../../data/core/systems_cleaned.csv")

results_folder = Path('./naive_errors/')
if not results_folder.is_dir():
    results_folder.mkdir()

In [2]:
# def naive_energy_forecaster(past_data: pd.DataFrame, times_to_predict: pd.DataFrame):
#     df = past_data.copy()
#     df['month_day'] = df['time'].dt.strftime('%m-%d')
#     df['time_of_day'] = df['time'].dt.time

#     avg_energy = (
#         df.groupby(['month_day', 'time_of_day'])['energy']
#         .mean()
#         .reset_index(name='energy_pred')
#     )

#     predictions = times_to_predict.copy()
#     predictions['month_day'] = predictions['time'].dt.strftime('%m-%d')
#     predictions['time_of_day'] = predictions['time'].dt.time
#     predictions = predictions.merge(
#         avg_energy,
#         on=['month_day', 'time_of_day'],
#         how='left'
#     )

#     predictions['energy_pred'] = predictions['energy_pred'].ffill().bfill()
    
#     return predictions['energy_pred']

In [3]:
# # experiment with system 4
# system_id=4
# check_prerun = PreRun(system_id=system_id, meter_or_inverter=None, path=read_path, systems_cleaned=systems_cleaned)
# check_prerun.fill_missing_hours()
# #do train test split
# #print("Good days:", check_prerun.good_days)
# good_days = check_prerun.good_days['date'].dt.date
# train_days = good_days[:int(0.8*len(good_days))]
# test_days = good_days[int(0.8*len(good_days)):]
# set_train_days = set(train_days)
# set_test_days = set(test_days)

# #print("check_prerun.data", check_prerun.data.head())

# train_data = check_prerun.data[check_prerun.data['time'].dt.date.isin(set_train_days)]
# test_data = check_prerun.data[check_prerun.data['time'].dt.date.isin(set_test_days)]

# # print("Train data:")
# # print(train_data.head())

# y_pred = naive_energy_forecaster(train_data, pd.DataFrame(test_data['time']))
# y_true = test_data['energy']

# #print(type(y_true.iloc[0]), type(y_pred.iloc[0]))



# print("custom error", PostRun.custom_error(y_true, y_pred, 1,2))


In [ ]:
system_reader_pairs = [(4,None),(10,None), (33, None), (50,None), (51,None), (1283,'inverter'),(1283,'meter')]
# system_reader_pairs = [(1283,'inverter'),(1283,'meter')]
for pair in system_reader_pairs:
    system_id = pair[0]
    reader_type = pair[1]

    check_prerun = PreRun(system_id=system_id, meter_or_inverter=reader_type, path=read_path, systems_cleaned=systems_cleaned)
    check_prerun.fill_missing_hours()
    system_recorded_max = check_prerun.data['energy'].max()
    check_prerun.good_end_days_naive(1)
    check_prerun.tts_of_data_using_end_days()
    all_data = check_prerun.amended_data.copy()

    pred_days = (check_prerun.end_days_naive['date']).dt.date
    pred_days_set=set(pred_days)

    #make the predictions! will be made as a new column of all_data
    all_data['key'] = list(zip(all_data['time'].dt.month, all_data['time'].dt.day, all_data['time'].dt.hour))
    all_data['naive_pred'] = (
        all_data.groupby('key')['energy'] #group by same month/day/time
        .transform(lambda x: x.expanding().mean().shift(1)) #expanding average, then shift by one to not include this year in calculation
    )

    #make sure value between 0 and highest observed max
    all_data['naive_pred'] = np.clip(all_data['naive_pred'], 0, system_recorded_max)

    all_data['naive_pred'] = all_data['naive_pred'].ffill().bfill()

    predicted_times = all_data.loc[all_data['time'].dt.date.isin(pred_days_set)]

    diff = predicted_times['energy'] - predicted_times['naive_pred']
    predicted_times['error'] = np.where(
        diff > 0,
        1 * diff**2,
        2 * diff**2
    )
    predicted_times['date'] = predicted_times['time'].dt.date
    daily_error = predicted_times.groupby('date')['error'].mean()
    
    # y_true = all_data.loc[all_data['time'].dt.date.isin(pred_days_set)][['time','energy']]
    # y_pred = all_data.loc[all_data['time'].dt.date.isin(pred_days_set)][['time','naive_pred']]
    # diff = y_true-y_pred

    # errors = pd.Series(
    #     np.where(diff > 0, 1 * diff**2, 2 * diff**2),
    #      index=diff.index
    # )
    
        

    daily_error.to_csv(f'naive_errors/{system_id}_{reader_type}_naive_errors.csv', index=False)


In [5]:
#compare hyperparameters
#System 4
print('System 4, None, Naive')
errors = pd.read_csv('naive_errors/4_None_naive_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 4, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 10, None, Naive with streak = 1')
errors = pd.read_csv('naive_errors/10_None_naive_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 10, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 33, None, Naive with streak = 1')
errors = pd.read_csv('naive_errors/33_None_naive_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 33, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 50, None, Naive with streak = 1')
errors = pd.read_csv('naive_errors/50_None_naive_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 50, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()


print('System 51, None, Naive with streak = 1')
errors = pd.read_csv('naive_errors/51_None_naive_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 51, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()



print('System 1283, Inverter, Naive with streak = 1')
errors = pd.read_csv('naive_errors/1283_inverter_naive_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 1283, meter_or_inverter = 'inverter', path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 1283, Meter, Naive with streak = 1')
errors = pd.read_csv('naive_errors/1283_meter_naive_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 1283, meter_or_inverter = 'meter', path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    



System 4, None, Naive
recorded system max: 1.0241429033333334
  Hyperparameters: error
     mean: 0.03790620181046905, median: 0.0168232768534236, min: 1.3703782109374944e-05, max: 0.3303032942815055, std: 0.047963462270966206

System 10, None, Naive with streak = 1
recorded system max: 1.185825
  Hyperparameters: error
     mean: 0.04459781009176729, median: 0.02022385796139095, min: 0.0001062334253333, max: 0.3235540966170364, std: 0.05722325438888556

System 33, None, Naive with streak = 1
recorded system max: 2.4044553333333334
  Hyperparameters: error
     mean: 0.20453918224081952, median: 0.09285378470533205, min: 0.0003345131093217, max: 1.5999259398978936, std: 0.2597563133058306

System 50, None, Naive with streak = 1
recorded system max: 7.072975
  Hyperparameters: error
     mean: 1.74772609323251, median: 0.630317534228296, min: 0.0029440071312421, max: 16.621730628500632, std: 2.641482804780033

System 51, None, Naive with streak = 1
recorded system max: 7.236874999999999

Let's take a look at the second half of the estimated stuff only

In [6]:
#compare hyperparameters
#System 4
print('System 4, None, Naive')
errors = pd.read_csv('naive_errors/4_None_naive_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 4, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 10, None, Naive with streak = 1')
errors = pd.read_csv('naive_errors/10_None_naive_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 10, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 33, None, Naive with streak = 1')
errors = pd.read_csv('naive_errors/33_None_naive_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 33, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 50, None, Naive with streak = 1')
errors = pd.read_csv('naive_errors/50_None_naive_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 50, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()


print('System 51, None, Naive with streak = 1')
errors = pd.read_csv('naive_errors/51_None_naive_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 51, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()



print('System 1283, Inverter, Naive with streak = 1')
errors = pd.read_csv('naive_errors/1283_inverter_naive_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 1283, meter_or_inverter = 'inverter', path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
print()

print('System 1283, Meter, Naive with streak = 1')
errors = pd.read_csv('naive_errors/1283_meter_naive_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 1283, meter_or_inverter = 'meter', path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    err=err.iloc[int(len(err)/2):]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    



System 4, None, Naive
recorded system max: 1.0241429033333334
  Hyperparameters: error
     mean: 0.022872264735096952, median: 0.0113990586644198, min: 0.0002167787939118, max: 0.2831337534140423, std: 0.030847311457254216

System 10, None, Naive with streak = 1
recorded system max: 1.185825
  Hyperparameters: error
     mean: 0.028386218023934196, median: 0.0134036526409713, min: 0.0003464478883116, max: 0.2940264823715728, std: 0.03857530699065842

System 33, None, Naive with streak = 1
recorded system max: 2.4044553333333334
  Hyperparameters: error
     mean: 0.130794638388839, median: 0.067660687032997, min: 0.0007001651694919, max: 1.57211261213771, std: 0.16931395821677186

System 50, None, Naive with streak = 1
recorded system max: 7.072975
  Hyperparameters: error
     mean: 0.9222170396110617, median: 0.4245000522281761, min: 0.0119598707423029, max: 9.759413479428066, std: 1.3130927610263448

System 51, None, Naive with streak = 1
recorded system max: 7.2368749999999995
  H

### Compare with 2nd half of linreg

Linreg better across the board